# Prepare data for network development model

This jupyter notebook contains all routines for the preparation of the input data sources into an input data file for the network development model 

**Authors:** Johannes Giehl (jfg.eco@cbs.dk)

#### import packages

In [1]:
import pandas as pd
import numpy as np
import os

#load the functions and methods from the corresponding file
from network_dev_data_preparation_functions import *

#### file paths

In [2]:
#set path to correct folders
#input data
input_file_path = os.path.join('..', '..', '01_data', '01_input_data', '01_raw')
#prepared input data
#output_file_path = '../01_data/01_input_data/02_processed/'
output_file_path = os.path.join('..', '..', '01_data', '01_input_data', '02_processed')
#file names

network_information = '\IGGIELGN_Network0.csv'
file_path_network_information  = input_file_path + network_information
full_path_network_information = os.path.abspath(os.path.join(os.getcwd(), file_path_network_information))

output_file_name = '\\network_data.xlsx'
output_path_file =  output_file_path + output_file_name
full_output_path_file = os.path.abspath(os.path.join(os.getcwd(), output_path_file))

input_network_data = '\input_network_data.xlsx'
input_network_data_path =  output_file_path + input_network_data
full_input_network_data_path = os.path.abspath(os.path.join(os.getcwd(), input_network_data_path))

#### Read in data

In [3]:
#import the network data
selected_columns = ['source', 'taget', 'capacity_Mm^3/d']
network_information_df = pd.read_csv(full_path_network_information, sep=';') #, sheet_name='Units', index_col=None)
# Replace column header of the capacity column
network_information_df.rename(columns={'capacity_Mm^3/d': 'capacity'}, inplace=True)
#end import data

In [4]:
#take the relevant parts of the input network data
coordinates_df = network_information_df[['source', 'target', 'capacity']]

In [21]:
source_name_counts = coordinates_df['source_name'].value_counts()
source_name_counts
# Specify the source_name you want to filter by
source_name_to_filter = 'node_0556'

# Filter the dataframe to show rows where 'source_name' is equal to source_name_to_filter
filtered_df = coordinates_df[coordinates_df['target_name'] == source_name_to_filter]

In [22]:
filtered_df

,source,target,capacity,source_name,target_name
679,"(1500967.5598713313, 6608671.121501834)","(1487951.9736882914, 6641885.125417695)",11.407561,node_0226,node_0556
680,"(1500967.5598713313, 6608671.121501834)","(1487951.9736882914, 6641885.125417695)",45.270756,node_0226,node_0556
708,"(1493615.6867408713, 6657711.430197078)","(1487951.9736882914, 6641885.125417695)",27.809143,node_0235,node_0556
709,"(1493615.6867408713, 6657711.430197078)","(1487951.9736882914, 6641885.125417695)",45.270756,node_0235,node_0556


In [33]:
network_complete_df
filtered_df2 = network_nodes_df[network_nodes_df['target_name'] == source_name_to_filter]
filtered_df2

,source_name,target_name,capacity
679,node_0226,node_0556,11.407561
680,node_0226,node_0556,45.270756
708,node_0235,node_0556,27.809143
709,node_0235,node_0556,45.270756


In [39]:
network_parallels_df
filtered_df3 = network_parallels_df[network_parallels_df['target_name'] == source_name_to_filter]
filtered_df3

,source_name,target_name
415,node_0556_1,node_0556
417,node_0556_2,node_0556
457,node_0556_1,node_0556
459,node_0556_2,node_0556


#### create parallel edges for model

In [37]:
def generate_parallel_connections(input_df):
    result_rows = []
    # Keep track of the occurrences for each source to target combination
    occurrence_tracker = {}

    for _, row in input_df.iterrows():
        source_name = row['source_name']
        target_name = row['target_name']
        occurrences = row['occurrences']

        # Initialize or update the occurrence count for the source to target combination
        if (source_name, target_name) not in occurrence_tracker:
            occurrence_tracker[(source_name, target_name)] = 1
        else:
            occurrence_tracker[(source_name, target_name)] += occurrences

        # Generate new target names based on the updated occurrence count
        for i in range(occurrence_tracker[(source_name, target_name)], occurrence_tracker[(source_name, target_name)] + occurrences):
            new_target = f"{target_name}_{i}"
            result_rows.append({'source_name': source_name, 'target_name': new_target})

            # Add a source node targeting the original target node
            result_rows.append({'source_name': new_target, 'target_name': target_name})

        # Update the occurrence count for the next iteration
        occurrence_tracker[(source_name, target_name)] += occurrences - 1

    result_df = pd.DataFrame(result_rows)

    return result_df

In [38]:
# Get unique entries from both columns
unique_entries = pd.unique(coordinates_df[['source', 'target']].values.ravel('K'))

# Create a dictionary to map unique entries to unique names
name_mapping = {entry: f'node_{i+1:04d}' for i, entry in enumerate(unique_entries)}

# Create a DataFrame with the pairs
unique_coordinate_node_df = pd.DataFrame({'Unique Entry': unique_entries, 'Unique Name': [name_mapping[entry] for entry in unique_entries]})

# Replace entries with unique names
coordinates_df['source_name'] = coordinates_df['source'].map(name_mapping)
coordinates_df['target_name'] = coordinates_df['target'].map(name_mapping)

# Display the result
network_nodes_df = coordinates_df[['source_name', 'target_name', 'capacity']]

# Count the occurrences of each source-target pair
pair_counts = network_nodes_df.groupby(['source_name', 'target_name']).size().reset_index(name='count')

# Filter pairs that exist more than once
duplicate_pairs = pair_counts[pair_counts['count'] > 1][['source_name', 'target_name']]
duplicate_pairs['occurrences'] = duplicate_pairs.apply(lambda row: network_nodes_df[(network_nodes_df['source_name'] == row['source_name']) & (network_nodes_df['target_name'] == row['target_name'])].shape[0], axis=1)

# Add capacity column to duplicate_pairs
network_parallels_df = generate_parallel_connections(duplicate_pairs)

# Sort by 'source_name' and move the nodes with a pattern like "node_0000_digit" to the end
network_parallels_df = (network_parallels_df.sort_values(by=['source_name', 'source_name'], key=lambda x: x.str.contains(r'node_\d{4}_\d+')))

# Identify entries with duplicates in either 'source_name' or 'target_name'
duplicates_mask = network_nodes_df.duplicated(subset=['source_name', 'target_name'], keep=False)

# Filter out rows with duplicates
single_network_edges_df = network_nodes_df[~duplicates_mask]

single_network_edges_df = single_network_edges_df[['source_name', 'target_name']]

C:\Users\jfg.eco\AppData\Local\Temp\ipykernel_15980\2315132568.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  coordinates_df['source_name'] = coordinates_df['source'].map(name_mapping)
C:\Users\jfg.eco\AppData\Local\Temp\ipykernel_15980\2315132568.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  coordinates_df['target_name'] = coordinates_df['target'].map(name_mapping)


In [25]:
#create unique index for the dataframe
# Combine the first two columns to create a new index
combined_indices = network_nodes_df['source_name'] + '_' + network_nodes_df['target_name']
counts = combined_indices.value_counts()

# Dictionary to keep track of the occurrences
occurrence_counter = counts.to_dict()

# Initialize the counter for duplicates
counter = {key: 0 for key in occurrence_counter.keys()}

# Apply the function to handle duplicates
unique_combined_indices = combined_indices.apply(handle_duplicates, occurrence_counter=occurrence_counter, counter=counter)
network_nodes_df['Combined'] = unique_combined_indices
network_nodes_df = network_nodes_df.set_index('Combined')

C:\Users\jfg.eco\AppData\Local\Temp\ipykernel_15980\1485637233.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  network_nodes_df['Combined'] = unique_combined_indices


#### final model network creation

In [26]:
#create the complete network and add parameters
network_complete_df = pd.concat([network_parallels_df, single_network_edges_df])
network_complete_df = network_complete_df.reset_index(drop=True)
# Sort by 'source_name' in ascending order
network_complete_df = network_complete_df.sort_values(by='target_name', ascending=True)
network_complete_df = network_complete_df.sort_values(by='source_name', ascending=True)


In [27]:
# Sort by 'source_name' and move the nodes with a pattern like "node_0000_digit" to the end
network_complete_df = (network_complete_df.sort_values(by=['source_name', 'source_name'], key=lambda x: x.str.contains(r'node_\d{4}_\d+')))


In [28]:
network_complete_df = network_complete_df.reset_index(drop=True)

network_complete_df['Combined'] = network_complete_df['source_name'] + '_' + network_complete_df['target_name'] 
network_complete_df = network_complete_df.set_index('Combined')


network_complete_df['capacity'] = network_nodes_df['capacity']
network_complete_df
# Replace NaN values in 'column1' with 0
network_complete_df = replace_nan_with_value(network_complete_df, 'capacity', 9999)


# Display the model_network_df DataFrame
network_complete_df.head(10)

,source_name,target_name,capacity
Combined,,,
node_0001_node_0002_2,node_0001,node_0002_2,27.809143
node_0001_node_0002_1,node_0001,node_0002_1,27.809143
node_0002_node_0362,node_0002,node_0362,65.753400
node_0003_node_0004,node_0003,node_0004,27.809143
node_0003_node_0196,node_0003,node_0196,6.329657
node_0003_node_0383,node_0003,node_0383,28.820865
node_0003_node_0381,node_0003,node_0381,27.809143
node_0003_node_0006_2,node_0003,node_0006_2,61.367897
node_0003_node_0382,node_0003,node_0382,27.809143


#### create model components information

In [10]:
#create a list with all nodes of the network
unique_nodes = np.array(pd.unique(network_complete_df[['source_name', 'target_name']].values.ravel('K')))
headers = 'nodes'
# Create a DataFrame
unique_nodes_df = pd.DataFrame(unique_nodes, columns = [headers])

#### export the data

In [11]:
#create the prepared input excel for the use in the network development model
with pd.ExcelWriter(full_output_path_file) as writer:
    unique_nodes_df.to_excel(writer, sheet_name='Nodes', index=False)
    network_complete_df.to_excel(writer, sheet_name='Edges', index=False)

In [12]:
#export the input network including the names of the nodes
with pd.ExcelWriter(full_input_network_data_path) as writer:
    coordinates_df.to_excel(writer, sheet_name='Network', index=False)